In [1]:
import sys
import time
import pandas as pd
import numpy as np
import pickle

from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from scipy.stats import ks_2samp
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from datetime import datetime
from sklearn.preprocessing import LabelBinarizer
from scipy.stats import randint as sp_randint
from scipy.stats import uniform as sp_uniform
from sklearn import metrics
#from matplotlib import pyplot as plt
#from sklearn_pandas import DataFrameMapper
import gc

# 缺失值：常量
DEFAULT = -999

# %matplotlib inline

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

import warnings
warnings.filterwarnings("ignore")
from openpyxl import Workbook, load_workbook

In [3]:
sys.path.append('/home/zengjunyao/notebook/to_zjy_模型训练/')

In [4]:
from train_tools.model_train_tools import *
from train_tools.in_file.in_file import *

In [30]:
label = pd.read_parquet('/home/zengjunyao/notebook/to_zjy_模型训练/af/df_all_sample_afwf.parquet')
print(label.shape)

(77619, 105)


In [31]:
label['sample_type'].value_counts()

sample_type
train    51422
oot       2347
Name: count, dtype: int64

In [5]:
module_list = [
    #'app_th_base_v1','app_th_base_v2','app_th_bcate_v1','app_th_cashloan_v1',
    'app_th_cate_v1','app_th_gcate_v1','app_th_gcate_v2','app_th_loan_v1','app_th_loan_v2','sms_th_base_v1','sms_th_base_v2','sms_th_cate_v1','sms_th_comp_v1','sms_th_loan_v1','sms_th_loan0_v1',
'app_th_woe_v1_a4','app_th_woe_v2_a4','sms_th_sender_woe_v1_a4','sms_th_woe_v1_a4','sms_th_woe_v2_a4']


In [6]:
module_list = [
    'app_id_base_v3',
    'app_id_comp_v1',
    'app_id_cate_v1',
    'app_id_bcate_v2',
    'app_id_cashloan_v2',
    'app_id_gcate_v3',
    'app_id_loan_v3',
    'dev_id_base_v1',
    'fdc_id_base_v2',
    'sms_id_base_v2',
    'sms_id_loan_v1',
    'ui_id_base_v2',
    'loan_id_base_v4',
    'fdc_id_plt_v1',
    'fdc_id_query_v1']

In [7]:
import sys
from train_tools.tools_iv_csi.feature_filter import filter_features_by_iv_csi
from train_tools.tools_iv_csi.iv_csi_calculator import IVCSICalculator
#from tools_woe.ModelTools import saveTrainValidWoe
#def emptyData(length):    #创建一个指定长度的空DataFrame，用于后续的数据填充或占位
 #   return pd.DataFrame([np.nan for i in range(length)])


In [8]:

# 创建IV和CSI计算器实例
calculator = IVCSICalculator(
    bin_num=10, 
    good_event=1, 
    process_missing=True, 
    n_jobs = 1
)

In [ ]:
for module in module_list:
    print(f"开始处理模块: {module}")
    # 读取数据文件
    module_files1 = glob.glob(f'/home/zengjunyao/notebook/model_link/af/{module}/df_1_*.parquet')
    module_files2 = glob.glob(f'/home/zengjunyao/notebook/model_link/af/{module}/df_2_*.parquet')
    module_files3 = glob.glob(f'/home/zengjunyao/notebook/model_link/wf/{module}/df_1_*.parquet')
    module_files4 = glob.glob(f'/home/zengjunyao/notebook/model_link/wf/{module}/df_2_*.parquet')
    #module_files2 = glob.glob(f'/home/mahaocheng/workspace/data/th_old/loan_features/{module}/df_0_*.parquet')
    #module_files3 = glob.glob(f'/home/mahaocheng/workspace/data/th_old2/loan_features/{module}/df_0_*.parquet')
    module_files = module_files1 + module_files2 + module_files3 + module_files4
    if not module_files:
        print(f" {module} 没有找到parquet文件")
        continue
    module_df = concat_parquet_files(module_files)
    
    # 只保留数值型（float/int）列，去掉完全恒定的列
    df_numeric = module_df.select_dtypes(include=['number']).copy()
    # 保留 app_order_id
    if 'app_order_id' in module_df.columns:
        df_numeric.insert(0, 'app_order_id', module_df['app_order_id'].values)
    nunique = df_numeric.nunique(dropna=False)
    valid_columns = nunique[nunique > 1].index
    module_df = df_numeric[valid_columns].reset_index(drop=True)
    # label = pd.read_pickle('/home/mahaocheng/workspace/model_train/model_train_result/th_0_all_1017/label_allsample_merge.pickle')
    module_df = pd.merge(module_df, label[['app_order_id','label','sample_type']], on='app_order_id', how='inner')

    # 分割数据集
    train_data, test_data = train_test_split(module_df[module_df['sample_type'] == 'train'], train_size=0.85, random_state=2023)
    valid_data = module_df[module_df['sample_type'] == 'oot'].reset_index(drop=True)
    train_data = train_data.reset_index(drop=True)
    test_data = test_data.reset_index(drop=True)

    primaryKey = ['app_order_id','sample_type']
    target = ["label"]

    feature_train_ = train_data.drop(primaryKey, axis=1)
    feature_test_ = test_data.drop(primaryKey, axis=1)
    feature_valid_ = valid_data.drop(primaryKey, axis=1)
    x_train = feature_train_.drop('label', axis=1)
    x_test = feature_test_.drop('label', axis=1)
    x_valid = feature_valid_.drop('label', axis=1)
    y_train = feature_train_[['label']]
    y_test = feature_test_[['label']]
    y_valid = feature_valid_[['label']]
    
    col_list = x_train.columns.to_list()
    print(f"特征数量: {len(col_list)}")
    
    # 逐个计算
    iv_results = []
    csi_results = []
    train_woes = {}
    valid_woes = {}
    
    for i, col in enumerate(col_list):
        # if i % 100 == 0:
        #     print(f"处理进度: {i}/{len(col_list)}")
        try:
            train_data_single = feature_train_[['label', col]].copy()
            valid_data_single = feature_valid_[['label', col]].copy()
            train_data_single.reset_index(drop=True, inplace=True)
            valid_data_single.reset_index(drop=True, inplace=True)
            
            train_iv, train_woe_dict = calculator.calculate_iv_woe(train_data_single, 'label', col)
            valid_iv, valid_woe_dict = calculator.calculate_iv_woe(valid_data_single, 'label', col)
            csi_value = calculator.calculate_csi(train_data_single, valid_data_single, col)
            
            iv_results.append({'feature': col, 'iv_train': train_iv, 'iv_valid': valid_iv})
            csi_results.append({'feature': col, 'CSI': csi_value})
            train_woes[col] = train_woe_dict
            valid_woes[col] = valid_woe_dict

        except Exception as e:
            print(f"特征 {col} 计算失败: {str(e)}")
            continue
    
    iv = pd.DataFrame(iv_results)
    csi = pd.DataFrame(csi_results)
    iv['ratio'] = iv['iv_valid'] / iv['iv_train']
    iv_csi = pd.merge(iv, csi, on='feature', how='left')
    iv_csi = iv_csi.sort_values(by=["iv_train"], ascending=False)
    # 保存iv_csi
    output_file = f'/home/zengjunyao/notebook/to_zjy_模型训练/model_train_result/{module}_iv_csi_{datetime.now().strftime("%Y%m%d")}.csv'
    iv_csi.to_csv(output_file, index=False, encoding='utf_8_sig')
    feature_list_new = filter_features_by_iv_csi(module_df, iv_csi)
    print(f'{module}筛选前特征数量：',len(col_list) ,'筛选后特征数量：',len(feature_list_new))
    module_df = module_df[['app_order_id','label','sample_type'] + feature_list_new]
    #保存筛选后特征文件
    module_df.to_parquet(f'/home/zengjunyao/notebook/to_zjy_模型训练/af/{module}_filtered_feature.parquet')
    print(f"模块 {module} 处理完成\n")
    


开始处理模块: app_id_base_v3


正在加载parquet文件: 100%|██████████| 860/860 [00:14<00:00, 58.97it/s]


成功合并 860 个文件，总行数: 85506
特征数量: 1048
处理进度: 0/1048
处理进度: 100/1048
处理进度: 200/1048
处理进度: 300/1048
处理进度: 400/1048
处理进度: 500/1048
处理进度: 600/1048
处理进度: 700/1048
处理进度: 800/1048
处理进度: 900/1048
处理进度: 1000/1048
app_id_base_v3筛选前特征数量： 1048 筛选后特征数量： 42
模块 app_id_base_v3 处理完成

开始处理模块: app_id_comp_v1


正在加载parquet文件: 100%|██████████| 860/860 [00:30<00:00, 28.54it/s]


成功合并 860 个文件，总行数: 85514
特征数量: 2246
处理进度: 0/2246
处理进度: 100/2246
处理进度: 200/2246
处理进度: 300/2246
处理进度: 400/2246
处理进度: 500/2246
处理进度: 600/2246
处理进度: 700/2246
处理进度: 800/2246
处理进度: 900/2246
处理进度: 1000/2246
处理进度: 1100/2246
处理进度: 1200/2246
处理进度: 1300/2246
处理进度: 1400/2246
处理进度: 1500/2246
处理进度: 1600/2246
处理进度: 1700/2246
处理进度: 1800/2246
处理进度: 1900/2246
处理进度: 2000/2246
处理进度: 2100/2246
处理进度: 2200/2246
成功合并 860 个文件，总行数: 85506
特征数量: 2420
处理进度: 0/2420
处理进度: 100/2420
处理进度: 200/2420
处理进度: 300/2420
处理进度: 400/2420
处理进度: 500/2420
处理进度: 600/2420
处理进度: 700/2420
处理进度: 800/2420
处理进度: 900/2420
处理进度: 1000/2420
处理进度: 1100/2420
处理进度: 1200/2420
处理进度: 1300/2420
处理进度: 1400/2420
处理进度: 1500/2420
处理进度: 1600/2420
处理进度: 1700/2420
处理进度: 1800/2420
处理进度: 1900/2420
处理进度: 2000/2420
处理进度: 2100/2420
处理进度: 2200/2420
处理进度: 2300/2420
处理进度: 2400/2420
app_id_cashloan_v2筛选前特征数量： 2420 筛选后特征数量： 9
模块 app_id_cashloan_v2 处理完成

开始处理模块: app_id_gcate_v3


...

成功合并 860 个文件，总行数: 85506
特征数量: 2272
处理进度: 0/2272
处理进度: 100/2272
处理进度: 200/2272
处理进度: 300/2272
处理进度: 400/2272
处理进度: 500/2272
处理进度: 600/2272
处理进度: 700/2272
处理进度: 800/2272
处理进度: 900/2272
处理进度: 1000/2272
处理进度: 1100/2272
处理进度: 1200/2272
处理进度: 1300/2272
处理进度: 1400/2272
处理进度: 1500/2272
处理进度: 1600/2272
处理进度: 1700/2272
处理进度: 1800/2272
处理进度: 1900/2272
处理进度: 2000/2272
处理进度: 2100/2272
处理进度: 2200/2272
app_id_loan_v3筛选前特征数量： 2272 筛选后特征数量： 25
模块 app_id_loan_v3 处理完成

开始处理模块: dev_id_base_v1


正在加载parquet文件: 100%|██████████| 860/860 [00:02<00:00, 397.65it/s]


成功合并 860 个文件，总行数: 85515
特征数量: 34
处理进度: 0/34
dev_id_base_v1筛选前特征数量： 34 筛选后特征数量： 3
模块 dev_id_base_v1 处理完成

开始处理模块: fdc_id_base_v2


正在加载parquet文件: 100%|██████████| 860/860 [07:45<00:00,  1.85it/s]


成功合并 860 个文件，总行数: 85415
特征数量: 32799
处理进度: 0/32799
处理进度: 100/32799
处理进度: 200/32799
处理进度: 300/32799
处理进度: 400/32799
处理进度: 500/32799
处理进度: 600/32799
处理进度: 700/32799
处理进度: 800/32799
处理进度: 900/32799
处理进度: 1000/32799
处理进度: 1100/32799
处理进度: 1200/32799
处理进度: 1300/32799
处理进度: 1400/32799
处理进度: 1500/32799
处理进度: 1600/32799
处理进度: 1700/32799
处理进度: 1800/32799
处理进度: 1900/32799
处理进度: 2000/32799
处理进度: 2100/32799
处理进度: 2200/32799
处理进度: 2300/32799
处理进度: 2400/32799
处理进度: 2500/32799
处理进度: 2600/32799
处理进度: 2700/32799
处理进度: 2800/32799
处理进度: 2900/32799
处理进度: 3000/32799
处理进度: 3100/32799
处理进度: 3200/32799
处理进度: 3300/32799
处理进度: 3400/32799
处理进度: 3500/32799
处理进度: 3600/32799
处理进度: 3700/32799
处理进度: 3800/32799
处理进度: 3900/32799
处理进度: 4000/32799
处理进度: 4100/32799
处理进度: 4200/32799
处理进度: 4300/32799
处理进度: 4400/32799
处理进度: 4500/32799
处理进度: 4600/32799
处理进度: 4700/32799
处理进度: 4800/32799
处理进度: 4900/32799
处理进度: 5000/32799
处理进度: 5100/32799
处理进度: 5200/32799
处理进度: 5300/32799
处理进度: 5400/32799
处理进度: 5500/32799
处理进度: 5600/32799
处理进度: 5

In [34]:
# 合并所有模块的特征数据
sample_list = [] 
for module in module_list:
    feature = pd.read_parquet(f'/home/zengjunyao/notebook/to_zjy_模型训练/af/{module}_filtered_feature.parquet')
    print(f'{module}：',feature.shape)
    sample_list.append(feature)

if len(sample_list) > 1:
    from functools import reduce
    sample = reduce(lambda left, right: pd.merge(left, right, on=['app_order_id','label','sample_type'], how='inner'), sample_list)
else:
    sample = sample_list[0].copy()
sample['sample_type'].fillna('train', inplace=True)
sample = sample.replace([np.inf, -np.inf], np.nan)
sample.fillna('-999', inplace=True)
print(sample.shape)
    

app_id_base_v3： (62985, 45)
app_id_comp_v1： (62992, 33)
app_id_cate_v1： (62992, 42)
app_id_bcate_v2： (62985, 5)
app_id_cashloan_v2： (62985, 12)
app_id_gcate_v3： (62985, 94)
app_id_loan_v3： (62985, 28)
dev_id_base_v1： (62993, 6)
fdc_id_base_v2： (62916, 1038)
sms_id_base_v2： (62985, 45)
sms_id_loan_v1： (62985, 13)
ui_id_base_v2： (62918, 6)
loan_id_base_v4： (62918, 349)
fdc_id_plt_v1： (62919, 3)
fdc_id_query_v1： (62914, 14)
(62913, 1691)


In [35]:
sample['sample_type'].value_counts()

sample_type
train    60568
oot       2345
Name: count, dtype: int64

In [36]:
print(sample.shape)
sample

(62913, 1691)


,app_order_id,label,sample_type,app_id_base_v3_install_mean_app_cnt_groupby_date_3d_7d_ratio,app_id_base_v3_install_day_cnt_15d,app_id_base_v3_update_day_cnt_15d_30d_ratio,app_id_base_v3_install_app_cnt_7d_15d_ratio,app_id_base_v3_install_var_app_cnt_groupby_date_60d_90d_ratio,app_id_base_v3_install_var_app_cnt_groupby_date_7d_15d_ratio,app_id_base_v3_install_app_cnt_60d_90d_ratio,app_id_base_v3_install_apply_last_days_diff_30d,app_id_base_v3_update_apply_first_days_diff_180d,app_id_base_v3_update_var_app_cnt_groupby_date_60d_90d_diff,app_id_base_v3_install_app_cnt_7d,app_id_base_v3_install_mean_app_cnt_groupby_date_3d,app_id_base_v3_install_avg_continuous_days_groupby_date_3d_7d_ratio,app_id_base_v3_install_mean_interval_day_groupby_date_7d,app_id_base_v3_install_first_last_days_diff_7d,app_id_base_v3_install_max_interval_day_groupby_date_180d,app_id_base_v3_install_time_density_7d,app_id_base_v3_install_app_cnt_3d_7d_ratio,app_id_base_v3_install_avg_continuous_days_groupby_date_7d_15d_ratio,app_id_base_v3_update_first_last_days_diff_90d,app_id_base_v3_update_day_cnt_30d_60d_ratio,app_id_base_v3_install_min_continuous_days_groupby_date_3d_7d_ratio,app_id_base_v3_install_day_cnt_infd,app_id_base_v3_update_mean_app_cnt_groupby_date_30d_60d_diff,app_id_base_v3_app_update_cnt,app_id_base_v3_update_day_cnt_60d_90d_ratio,app_id_base_v3_install_day_cnt_180d_infd_ratio,app_id_base_v3_update_time_density_180d,app_id_base_v3_update_day_cnt_infd,app_id_base_v3_install_max_app_cnt_groupby_date_3d_7d_ratio,app_id_base_v3_install_median_app_cnt_groupby_date_3d_7d_ratio,app_id_base_v3_install_median_continuous_days_groupby_date_7d_15d_ratio,app_id_base_v3_install_apply_last_days_diff_15d,app_id_base_v3_install_var_app_cnt_groupby_date_15d,app_id_base_v3_install_time_density_60d_90d_ratio,app_id_base_v3_update_apply_first_days_diff_60d_90d_ratio,app_id_base_v3_install_day_cnt_30d_60d_ratio,app_id_base_v3_update_mean_interval_day_groupby_date_30d_60d_ratio,app_id_base_v3_install_apply_last_days_diff_60d,app_id_base_v3_install_mean_app_cnt_groupby_date_7d,app_id_base_v3_install_app_cnt_3d,app_id_base_v3_update_app_cnt_30d_60d_ratio,app_id_comp_v1_seinst_updt_nonpreinst_iscomp_instdelmaxnull30_apply_start_diff_d7_0,app_id_comp_v1_seupdt_inst_iscomp_updt_end_start_diff_all_0,app_id_comp_v1_seupdt_inst_nonpreinst_iscomp_updt_end_start_diff_all_0,app_id_comp_v1_seinst_updt_iscomp_inst_day_avg_cnt_m6_0,app_id_comp_v1_up2in_updt_iscomp_up2in_time_diff_days_medium_d15_0,...,loan_id_base_v4_other_product_apply_overdue_days_std_in_inf_d,loan_id_base_v4_brush_order_third_loan_first_repay_time_diff,loan_id_base_v4_all_loan_success_inter_mean_rate_in_inf_d,loan_id_base_v4_self_app_pre_payoff_diff_mean_in_15_d,loan_id_base_v4_self_app_pre_payoff_diff_mean_in_15_orders,loan_id_base_v4_other_product_pre_payoff_overdue_days_mean_rate_in_15_orders,loan_id_base_v4_all_pre_payoff_inter_sum_in_15_orders,loan_id_base_v4_all_pre_payoff_overdue_days_mean_rate_in_15_d,loan_id_base_v4_other_app_pass_max_length_in_90_d,loan_id_base_v4_all_payoff_diff_mean_rate_in_15_orders,loan_id_base_v4_self_app_pre_payoff_diff_min_in_agg_180d,loan_id_base_v4_self_app_apply_diff_mean_in_15_orders,loan_id_base_v4_other_product_pre_payoff_diff_min_in_agg_90d,loan_id_base_v4_self_app_pre_payoff_inter_sum_in_7_orders,loan_id_base_v4_self_app_pre_payoff_diff_std_in_7_orders,loan_id_base_v4_all_pre_payoff_inter_mean_rate_in_30_d,loan_id_base_v4_other_product_pre_payoff_diff_range_in_7_orders,loan_id_base_v4_other_product_pre_payoff_overdue_days_mean_rate_in_90_d,loan_id_base_v4_all_payoff_inter_min_in_inf_d,loan_id_base_v4_self_app_pre_payoff_inter_mean_rate_in_7_d,loan_id_base_v4_all_pass_diff_mean_in_180_d,loan_id_base_v4_other_product_pre_payoff_inter_sum_in_7_orders,loan_id_base_v4_all_pre_payoff_overdue_days_mean_rate_in_30_d,loan_id_base_v4_brush_order_first_repay_early_day,loan_id_base_v4_all_loan_success_max_length_in_15_orders,loan_id_base_v4_self_app_payoff_diff_mean_in_agg_30d,l

In [37]:
#切分训练集测试集、oot
train_data,test_data = train_test_split(sample[sample['sample_type'] == 'train'],train_size=0.85,random_state=2023)
valid_data = sample[sample['sample_type'] == 'oot']

In [38]:
#分别统计oot和train的好坏样本情况
#将sample按照sample_type(oot/train)进行分组聚合（oot与train各为一组），再对分组后的数据对每组的label列进行聚合操作：
sample.groupby(["sample_type"],as_index =False).label.agg({"total":"count",  #统计每组总数，并将结果命名为total。
                                                           "good":lambda x:(x==0).sum(),   #统计每组label值为0的样本数量（未逾期），命名good。
                                                           "bad":lambda x:(x==1).sum(),     #统计每组label值为1的样本数量（逾期），命名 bad。
                                                           "nosure":lambda x:(x==2).sum(),    #统计每组label值为2的样本数量，命名nosure。
                                                           "bad_rate":lambda x:round((x==1).sum()/x.count(),4)})#计算坏样本率，命名bad_rate

,sample_type,total,good,bad,nosure,bad_rate
0,oot,2345,1564,781,0,0.3330
1,train,60568,40138,20430,0,0.3373


In [39]:
#样本预处理（重置索引）
train_data=train_data.reset_index(drop=True)
test_data=test_data.reset_index(drop=True)
valid_data=valid_data.reset_index(drop=True)

print('训练集：',train_data.shape)
print('测试集：',test_data.shape)
print('验证集：',valid_data.shape)

train_data.head(2)#查看前两行

训练集： (51482, 1691)
测试集： (9086, 1691)
验证集： (2345, 1691)


,app_order_id,label,sample_type,app_id_base_v3_install_mean_app_cnt_groupby_date_3d_7d_ratio,app_id_base_v3_install_day_cnt_15d,app_id_base_v3_update_day_cnt_15d_30d_ratio,app_id_base_v3_install_app_cnt_7d_15d_ratio,app_id_base_v3_install_var_app_cnt_groupby_date_60d_90d_ratio,app_id_base_v3_install_var_app_cnt_groupby_date_7d_15d_ratio,app_id_base_v3_install_app_cnt_60d_90d_ratio,app_id_base_v3_install_apply_last_days_diff_30d,app_id_base_v3_update_apply_first_days_diff_180d,app_id_base_v3_update_var_app_cnt_groupby_date_60d_90d_diff,app_id_base_v3_install_app_cnt_7d,app_id_base_v3_install_mean_app_cnt_groupby_date_3d,app_id_base_v3_install_avg_continuous_days_groupby_date_3d_7d_ratio,app_id_base_v3_install_mean_interval_day_groupby_date_7d,app_id_base_v3_install_first_last_days_diff_7d,app_id_base_v3_install_max_interval_day_groupby_date_180d,app_id_base_v3_install_time_density_7d,app_id_base_v3_install_app_cnt_3d_7d_ratio,app_id_base_v3_install_avg_continuous_days_groupby_date_7d_15d_ratio,app_id_base_v3_update_first_last_days_diff_90d,app_id_base_v3_update_day_cnt_30d_60d_ratio,app_id_base_v3_install_min_continuous_days_groupby_date_3d_7d_ratio,app_id_base_v3_install_day_cnt_infd,app_id_base_v3_update_mean_app_cnt_groupby_date_30d_60d_diff,app_id_base_v3_app_update_cnt,app_id_base_v3_update_day_cnt_60d_90d_ratio,app_id_base_v3_install_day_cnt_180d_infd_ratio,app_id_base_v3_update_time_density_180d,app_id_base_v3_update_day_cnt_infd,app_id_base_v3_install_max_app_cnt_groupby_date_3d_7d_ratio,app_id_base_v3_install_median_app_cnt_groupby_date_3d_7d_ratio,app_id_base_v3_install_median_continuous_days_groupby_date_7d_15d_ratio,app_id_base_v3_install_apply_last_days_diff_15d,app_id_base_v3_install_var_app_cnt_groupby_date_15d,app_id_base_v3_install_time_density_60d_90d_ratio,app_id_base_v3_update_apply_first_days_diff_60d_90d_ratio,app_id_base_v3_install_day_cnt_30d_60d_ratio,app_id_base_v3_update_mean_interval_day_groupby_date_30d_60d_ratio,app_id_base_v3_install_apply_last_days_diff_60d,app_id_base_v3_install_mean_app_cnt_groupby_date_7d,app_id_base_v3_install_app_cnt_3d,app_id_base_v3_update_app_cnt_30d_60d_ratio,app_id_comp_v1_seinst_updt_nonpreinst_iscomp_instdelmaxnull30_apply_start_diff_d7_0,app_id_comp_v1_seupdt_inst_iscomp_updt_end_start_diff_all_0,app_id_comp_v1_seupdt_inst_nonpreinst_iscomp_updt_end_start_diff_all_0,app_id_comp_v1_seinst_updt_iscomp_inst_day_avg_cnt_m6_0,app_id_comp_v1_up2in_updt_iscomp_up2in_time_diff_days_medium_d15_0,...,loan_id_base_v4_other_product_apply_overdue_days_std_in_inf_d,loan_id_base_v4_brush_order_third_loan_first_repay_time_diff,loan_id_base_v4_all_loan_success_inter_mean_rate_in_inf_d,loan_id_base_v4_self_app_pre_payoff_diff_mean_in_15_d,loan_id_base_v4_self_app_pre_payoff_diff_mean_in_15_orders,loan_id_base_v4_other_product_pre_payoff_overdue_days_mean_rate_in_15_orders,loan_id_base_v4_all_pre_payoff_inter_sum_in_15_orders,loan_id_base_v4_all_pre_payoff_overdue_days_mean_rate_in_15_d,loan_id_base_v4_other_app_pass_max_length_in_90_d,loan_id_base_v4_all_payoff_diff_mean_rate_in_15_orders,loan_id_base_v4_self_app_pre_payoff_diff_min_in_agg_180d,loan_id_base_v4_self_app_apply_diff_mean_in_15_orders,loan_id_base_v4_other_product_pre_payoff_diff_min_in_agg_90d,loan_id_base_v4_self_app_pre_payoff_inter_sum_in_7_orders,loan_id_base_v4_self_app_pre_payoff_diff_std_in_7_orders,loan_id_base_v4_all_pre_payoff_inter_mean_rate_in_30_d,loan_id_base_v4_other_product_pre_payoff_diff_range_in_7_orders,loan_id_base_v4_other_product_pre_payoff_overdue_days_mean_rate_in_90_d,loan_id_base_v4_all_payoff_inter_min_in_inf_d,loan_id_base_v4_self_app_pre_payoff_inter_mean_rate_in_7_d,loan_id_base_v4_all_pass_diff_mean_in_180_d,loan_id_base_v4_other_product_pre_payoff_inter_sum_in_7_orders,loan_id_base_v4_all_pre_payoff_overdue_days_mean_rate_in_30_d,loan_id_base_v4_brush_order_first_repay_early_day,loan_id_base_v4_all_loan_success_max_length_in_15_orders,loan_id_base_v4_self_app_payoff_diff_mean_in_agg_30d,l

4.特征筛选

In [40]:
primaryKey=['app_order_id','sample_type'] #定义主键
target = ["label"]  #定义target

### 删除主键，只留target变量与特征变量x
feature_train_ = train_data.drop(primaryKey,axis=1)
feature_test_ = test_data.drop(primaryKey,axis=1)
feature_valid_ = valid_data.drop(primaryKey,axis=1)

### 分离x特征变量
x_train = feature_train_.drop('label',axis=1)
x_test = feature_test_.drop('label',axis=1)
x_valid = feature_valid_.drop('label',axis=1)

### 分离目标变量y标签target
y_train = feature_train_[['label']]
y_test = feature_test_[['label']]
y_valid = feature_valid_[['label']]

print(x_train.shape)   #shape返回x_train的行数和列数

(51482, 1688)


# 最终筛选加调参：手工

In [41]:
%%time
model = LGBMClassifier(
    #Boosting Parameters参数
    objective='binary',         #binary:logistic – 二分类逻辑回归，输出为概率
    #subsample=0.8,              #bagging_fraction，训练每个决策树所用到的子样本占总样本的比例；参考：0.8-1之间
    learning_rate=0.04,          #shrinkage rate 每次学习的步长. 越小模型越稳健  参考：[0.01, 0.02, 0.05, 0.1, 0.15,0.2]
    
    # other参数
    reg_alpha=0, 
    reg_lambda=0,  #L1 L2正则化项权重.可预防过拟合；
    #scale_pos_weight=1,         #正样本的权重 默认为1； trick如果要加强负样本权重，此值设置<1??;
    seed=23,                    #随机种子
    #is_unbalance=False,
    #sigmoid=1.0,                #sigmoid函数的参数,将用作于二分类;
    #subsample_for_bin=1000,     #构造分箱的样本数  默认50000;  手工1000 5000 10000 20000？？
    #subsample_freq=2,           #同一样本最大选择次数; 手工1 or 2？？
    #uniform_drop=False,         #only used in dart, true if want to use uniform drop
    #skip_drop=0.5,              #only used in dart, probability of skipping drop
    #xgboost_dart_mode=False,    #only used in dart, true if want to use xgboost dart mode
    #drop_rate=0.6,              #default=0.1, type=double,only used in dart
    verbose=-1,
    
    #Tree Parameters参数
    #max_bin=20,               #每个特征的最大分箱数. 默认255，手工加大？？
    max_depth=3,              #树的最大深度,一般5-8，推荐GBDT树的深度：6
    num_leaves=7,            #所有树最大叶子节点数. 默认=31
    #max_drop=5,               #最大叶子节点drop样本数  手工加大？？
    #colsample_bytree=0.8,     #构造每棵树时的子样本抽样比例.  0.7-0.8
    #min_child_samples=200,    #构建一个叶子节点所需的最少样本量，alias=min_data_per_leaf , min_data, min_data_in_leaf. default=20  手工减小？？
    #min_child_weight=3,       #一个叶子节点所需最小权重和，默认0.001
    #min_split_gain=3.5,       #在叶子节点做进一步划分的最小损失，默认0
    n_estimators=360,         #决策树的数量  1500-2000
    )
model.fit(x_train,y_train)
result,predprob_train,predprob_test,predprob_valid=getKSresult(model,x_train,y_train,x_test,y_test,x_valid,y_valid)
result

CPU times: user 5min 30s, sys: 233 ms, total: 5min 31s
Wall time: 15 s


,train_ks,test_ks,valid_ks,overfitting,loss
0,0.420497,0.36102,0.261234,14.144519,37.875015


In [49]:
### 变量特征重要性
feature_importance = get_feature_importance(model, x_train, importance_type='gain')
feature_importance = feature_importance.reset_index()
feature_importance.columns = ['imp','feature']
print('筛选前特征：',feature_importance.shape)
feature_importance_final = feature_importance[:680]
print('筛选后特征：',feature_importance_final.shape)
feature_importance_final.tail(10)

筛选前特征： (1688, 2)
筛选后特征： (680, 2)


,imp,feature
670,16.201500,loan_id_base_v4_all_deny_diff_mean_in_agg_15d
671,15.711300,app_id_base_v3_install_var_app_cnt_groupby_dat...
672,15.166580,fdc_id_base_v2_early_settle_trans_amt_min_d90
673,13.211100,fdc_id_base_v2_remain_amount_over90_heavy_over...
674,11.399600,loan_id_base_v4_self_app_deny_diff_mean_in_15_d
675,9.555700,loan_id_base_v4_self_app_pre_payoff_inter_max_...
676,1.583180,fdc_id_base_v2_large_trans_amount_over1m_earli...
677,1.410030,loan_id_base_v4_other_product_pre_payoff_amoun...
678,0.506312,sms_id_base_v2_all_phone_var_body_discnt_group...
679,0.094585,fdc_id_base_v2_install_loan_trans_amt_sum_rati...


In [50]:
feature_list_final = list(set(feature_importance_final.feature))
# feature_list_final = DF_columns

print(len(feature_list_final))

train_data_v1 = train_data[['label']+feature_list_final]
test_data_v1 = test_data[['label']+feature_list_final]
valid_data_v1 = valid_data[['label']+feature_list_final]

### x变量
x_train = train_data_v1.drop('label',axis=1)
x_test = test_data_v1.drop('label',axis=1)
x_valid = valid_data_v1.drop('label',axis=1)

### y标签
y_train = train_data_v1[['label']]
y_test = test_data_v1[['label']]
y_valid = valid_data_v1[['label']]
print(train_data_v1.shape)

680
(51482, 681)


In [51]:
%%time
model = LGBMClassifier(
    #Boosting Parameters参数
    objective='binary',          #binary:logistic – 二分类逻辑回归，输出为概率
    subsample=1,              #bagging_fraction，训练每个决策树所用到的子样本占总样本的比例；参考：0.8-1之间
    learning_rate=0.06,          #shrinkage rate 每次学习的步长. 越小模型越稳健  参考：[0.01, 0.02, 0.05, 0.1, 0.15,0.2]

    # other参数
    reg_alpha=10, 
    reg_lambda=10,                #L1 L2正则化项权重.可预防过拟合；
    scale_pos_weight=1,          #正样本的权重 默认为1； trick如果要加强负样本权重，此值设置<1??;
    seed=2024,                    #随机种子
    # is_unbalance=True,
    # sigmoid=1.0,                #sigmoid函数的参数,将用作于二分类;
    # subsample_for_bin=1000,     #构造分箱的样本数  默认50000;  手工1000 5000 10000 20000？？
    # subsample_freq=2,           #同一样本最大选择次数; 手工1 or 2？？
    # uniform_drop=False,         #only used in dart, true if want to use uniform drop
    # skip_drop=0.5,              #only used in dart, probability of skipping drop
    # xgboost_dart_mode=False,    #only used in dart, true if want to use xgboost dart mode
    # drop_rate=0.6,              #default=0.1, type=double,only used in dart
    verbose=-1,

    #Tree Parameters参数
    #max_bin=200,               #每个特征的最大分箱数. 默认255，手工加大？？
    max_depth=2,              #树的最大深度,一般5-8，推荐GBDT树的深度：6
    num_leaves=2,            #所有树最大叶子节点数. 默认=31
    #max_drop=5,               #最大叶子节点drop样本数  手工加大？？
    colsample_bytree=0.8,     #构造每棵树时的子样本抽样比例.  0.7-0.8
    min_child_samples = 100,    #构建一个叶子节点所需的最少样本量，alias=min_data_per_leaf , min_data, min_data_in_leaf. default=20  手工减小？？
    #min_child_weight=3,       #一个叶子节点所需最小权重和，默认0.001
    #min_split_gain=3.5,       #在叶子节点做进一步划分的最小损失，默认0
    n_estimators=480,         #决策树的数量  1500-2000
)
model.fit(x_train,y_train)
result,predprob_train,predprob_test,predprob_valid=getKSresult(model,x_train,y_train,x_test,y_test,x_valid,y_valid)
result

CPU times: user 1min 35s, sys: 0 ns, total: 1min 35s
Wall time: 4.45 s


,train_ks,test_ks,valid_ks,overfitting,loss
0,0.28603,0.277089,0.250907,3.126051,12.279542


In [57]:
### 变量特征重要性
feature_importance_v2 = get_feature_importance(model, x_train, importance_type='gain')
feature_importance_v2 = feature_importance_v2.reset_index()
feature_importance_v2.columns = ['imp','feature']
print(feature_importance_v2.shape)
feature_importance_final = feature_importance_v2[:125]   #筛选出前n个重要性最高的特征
print(feature_importance_final.shape)
feature_importance_final.tail(10)

(680, 2)
(125, 2)


,imp,feature
115,27.452101,app_id_gcate_v3_update_game_min_mininstalls_18...
116,26.760900,loan_id_base_v4_other_app_apply_cnt_in_7_d
117,26.649500,fdc_id_base_v2_remain_amount_over80_light_over...
118,26.330500,fdc_id_base_v2_tadpole_loan_trans_amt_max_rati...
119,25.820601,fdc_id_base_v2_loan_days_over60d_least_trans_p...
120,25.610100,app_id_cate_v1_mininstalls_avg_y1
121,25.528000,fdc_id_base_v2_new_mob_plt_remain_amt_min_d30
122,25.475599,fdc_id_base_v2_large_trans_amount_over200k_ins...
123,25.455200,fdc_id_base_v2_large_trans_amount_over2m_least...
124,25.415800,fdc_id_base_v2_remain_amount_over90_loan_level...


In [58]:
feature_list_final = list(set(feature_importance_final.feature))
# feature_list_final = DF_columns

print(len(feature_list_final))

train_data_v1 = train_data[['label']+feature_list_final]
test_data_v1 = test_data[['label']+feature_list_final]
valid_data_v1 = valid_data[['label']+feature_list_final]

### x变量
x_train = train_data_v1.drop('label',axis=1)
x_test = test_data_v1.drop('label',axis=1)
x_valid = valid_data_v1.drop('label',axis=1)

### y标签
y_train = train_data_v1[['label']]
y_test = test_data_v1[['label']]
y_valid = valid_data_v1[['label']]
print(train_data_v1.shape)

125
(51482, 126)


In [59]:
%%time
model = LGBMClassifier(
    #Boosting Parameters参数
    objective='binary',          #binary:logistic – 二分类逻辑回归，输出为概率
    subsample=1,              #bagging_fraction，训练每个决策树所用到的子样本占总样本的比例；参考：0.8-1之间
    learning_rate=0.06,          #shrinkage rate 每次学习的步长. 越小模型越稳健  参考：[0.01, 0.02, 0.05, 0.1, 0.15,0.2]

    # other参数
    reg_alpha=10, 
    reg_lambda=10,                #L1 L2正则化项权重.可预防过拟合；
    scale_pos_weight=1,          #正样本的权重 默认为1； trick如果要加强负样本权重，此值设置<1??;
    seed=2024,                    #随机种子
    # is_unbalance=True,
    # sigmoid=1.0,                #sigmoid函数的参数,将用作于二分类;
    # subsample_for_bin=1000,     #构造分箱的样本数  默认50000;  手工1000 5000 10000 20000？？
    # subsample_freq=2,           #同一样本最大选择次数; 手工1 or 2？？
    # uniform_drop=False,         #only used in dart, true if want to use uniform drop
    # skip_drop=0.5,              #only used in dart, probability of skipping drop
    # xgboost_dart_mode=False,    #only used in dart, true if want to use xgboost dart mode
    # drop_rate=0.6,              #default=0.1, type=double,only used in dart
    verbose=-1,

    #Tree Parameters参数
    #max_bin=200,               #每个特征的最大分箱数. 默认255，手工加大？？
    max_depth=2,              #树的最大深度,一般5-8，推荐GBDT树的深度：6
    num_leaves=2,            #所有树最大叶子节点数. 默认=31
    #max_drop=5,               #最大叶子节点drop样本数  手工加大？？
    colsample_bytree=0.8,     #构造每棵树时的子样本抽样比例.  0.7-0.8
    min_child_samples = 100,    #构建一个叶子节点所需的最少样本量，alias=min_data_per_leaf , min_data, min_data_in_leaf. default=20  手工减小？？
    #min_child_weight=3,       #一个叶子节点所需最小权重和，默认0.001
    #min_split_gain=3.5,       #在叶子节点做进一步划分的最小损失，默认0
    n_estimators=480,         #决策树的数量  1500-2000
)
model.fit(x_train,y_train)
result,predprob_train,predprob_test,predprob_valid=getKSresult(model,x_train,y_train,x_test,y_test,x_valid,y_valid)
result

CPU times: user 25.3 s, sys: 0 ns, total: 25.3 s
Wall time: 1.07 s


,train_ks,test_ks,valid_ks,overfitting,loss
0,0.286048,0.275105,0.248353,3.825509,13.177994


### 网格调参

In [60]:
#第一次参数初始范围
max_depth_list =         [2]
num_leaves_list =        [2,3]
min_child_samples_list = [100]
reg_alpha_list  =    [0,10,20,30,40]
reg_lambda_list =    [0,10,20,30,40]
learning_rate_list = [0.03,0.04,0.05]
n_estimators_list =  [300,400,500]
print(f'max_depth_list:{len(max_depth_list)}  {max_depth_list},num_leaves_list:{len(num_leaves_list)} {num_leaves_list}')
print(f'min_child_samples_list:{len(min_child_samples_list)} {min_child_samples_list}')
print(f'max_depth_list:{len(reg_alpha_list)}  {reg_alpha_list},reg_lambda_list:{len(reg_lambda_list)} {reg_lambda_list}')
print(f'learning_rate_list:{len(learning_rate_list)}  {learning_rate_list},n_estimators_list:{len(n_estimators_list)} {n_estimators_list}')

total_times = len(max_depth_list)*len(num_leaves_list)*len(min_child_samples_list)*len(reg_alpha_list)*len(reg_lambda_list)*len(learning_rate_list)*len(n_estimators_list)
print('网格次数:',total_times)

one_times_s = 1.4#秒

print('网格总耗时(小时):',round(total_times*one_times_s/3600,2))

max_depth_list:1  [2],num_leaves_list:2 [2, 3]
min_child_samples_list:1 [100]
max_depth_list:5  [0, 10, 20, 30, 40],reg_lambda_list:5 [0, 10, 20, 30, 40]
learning_rate_list:3  [0.03, 0.04, 0.05],n_estimators_list:3 [300, 400, 500]
网格次数: 450
网格总耗时(小时): 0.17


In [61]:
%%time
#循环调参：max_depth num_leaves
ks_result=[]
i= 1
for max_depth in max_depth_list:
    for num_leaves in num_leaves_list:
        for min_child_samples in min_child_samples_list:
            for reg_alpha in reg_alpha_list:
                for reg_lambda in reg_lambda_list:
                    for learning_rate in learning_rate_list:
                        for n_estimators in n_estimators_list:
                            model = LGBMClassifier(
                                #Boosting Parameters参数
                                objective='binary',          #binary:logistic – 二分类逻辑回归，输出为概率
                                subsample=1,              #bagging_fraction，训练每个决策树所用到的子样本占总样本的比例；参考：0.8-1之间
                                learning_rate=learning_rate,          #shrinkage rate 每次学习的步长. 越小模型越稳健  参考：[0.01, 0.02, 0.05, 0.1, 0.15,0.2]

                                # other参数
                                reg_alpha=reg_alpha, 
                                reg_lambda=reg_lambda,                #L1 L2正则化项权重.可预防过拟合；
                                scale_pos_weight=1,          #正样本的权重 默认为1； trick如果要加强负样本权重，此值设置<1??;
                                seed=2024,                    #随机种子
                                # is_unbalance=True,
                                # sigmoid=1.0,                #sigmoid函数的参数,将用作于二分类;
                                # subsample_for_bin=1000,     #构造分箱的样本数  默认50000;  手工1000 5000 10000 20000？？
                                # subsample_freq=2,           #同一样本最大选择次数; 手工1 or 2？？
                                # uniform_drop=False,         #only used in dart, true if want to use uniform drop
                                # skip_drop=0.5,              #only used in dart, probability of skipping drop
                                # xgboost_dart_mode=False,    #only used in dart, true if want to use xgboost dart mode
                                # drop_rate=0.6,              #default=0.1, type=double,only used in dart
                                verbose=-1,

                                #Tree Parameters参bb
                                #max_bin=200,               #每个特征的最大分箱数. 默认255，手工加大？？
                                max_depth=max_depth,              #树的最大深度,一般5-8，推荐GBDT树的深度：6
                                num_leaves=num_leaves,            #所有树最大叶子节点数. 默认=31
                                #max_drop=5,               #最大叶子节点drop样本数  手工加大？？
                                colsample_bytree=0.8,     #构造每棵树时的子样本抽样比例.  0.7-0.8
                                min_child_samples = min_child_samples,    #构建一个叶子节点所需的最少样本量，alias=min_data_per_leaf , min_data, min_data_in_leaf. default=20  手工减小？？
                                #min_child_weight=3,       #一个叶子节点所需最小权重和，默认0.001
                                #min_split_gain=3.5,       #在叶子节点做进一步划分的最小损失，默认0
                                n_estimators=n_estimators,         #决策树的数量  1500-2000
                            )
                            model.fit(x_train,y_train)
                            result,predprob_train,predprob_test,predprob_valid=getKSresult(model,x_train,y_train,x_test,y_test,x_valid,y_valid)
                            result['max_depth']=max_depth
                            result['num_leaves']=num_leaves
                            result['min_child_samples']=min_child_samples
                            result['reg_alpha']=reg_alpha
                            result['reg_lambda']=reg_lambda
                            result['learning_rate']=learning_rate
                            result['n_estimators']=n_estimators
                            ks_result.append(result)
                            print(f"第{i}次，参数: max_depth:{max_depth},num_leaves:{num_leaves},min_child_samples:{min_child_samples},reg_alpha:{reg_alpha},reg_lambda:{reg_lambda},learning_rate:{learning_rate},n_estimators:{n_estimators}; 模型KS: train_ks:{result['train_ks'].iloc[0]},test_ks:{result['test_ks'].iloc[0]},valid_ks:{result['valid_ks'].iloc[0]}")
                            i= i+1
#将ks_result结果保存为df_ks_result
df_ks_result = pd.DataFrame()
for result in ks_result: 
    df_ks_result = pd.concat([df_ks_result,result])

第1次，参数: max_depth:2,num_leaves:2,min_child_samples:100,reg_alpha:0,reg_lambda:0,learning_rate:0.03,n_estimators:300; 模型KS: train_ks:0.24381835748494912,test_ks:0.24430672671375964,valid_ks:0.2598273902891892
第2次，参数: max_depth:2,num_leaves:2,min_child_samples:100,reg_alpha:0,reg_lambda:0,learning_rate:0.03,n_estimators:400; 模型KS: train_ks:0.25429075178354743,test_ks:0.25086816906034004,valid_ks:0.256641920811079
第3次，参数: max_depth:2,num_leaves:2,min_child_samples:100,reg_alpha:0,reg_lambda:0,learning_rate:0.03,n_estimators:500; 模型KS: train_ks:0.26173427609958827,test_ks:0.25507521597795974,valid_ks:0.26249136296504905
第4次，参数: max_depth:2,num_leaves:2,min_child_samples:100,reg_alpha:0,reg_lambda:0,learning_rate:0.04,n_estimators:300; 模型KS: train_ks:0.25547636717129657,test_ks:0.25153488032646804,valid_ks:0.25918472939473625
第5次，参数: max_depth:2,num_leaves:2,min_child_samples:100,reg_alpha:0,reg_lambda:0,learning_rate:0.04,n_estimators:400; 模型KS: train_ks:0.2655306257973721,test_ks:0.257134

In [62]:
#将ks_result结果保存为df_ks_result
df_ks_result = pd.DataFrame()
for result in ks_result: 
    df_ks_result = pd.concat([df_ks_result,result])

In [63]:
df_ks_result.sort_values(by = 'valid_ks',ascending=False)[:5]

,train_ks,test_ks,valid_ks,overfitting,loss,max_depth,num_leaves,min_child_samples,reg_alpha,reg_lambda,learning_rate,n_estimators
0,0.273544,0.258825,0.265660,5.380955,2.881960,2,2,100,0,10,0.05,400
0,0.268430,0.258189,0.264393,3.814998,1.503748,2,2,100,30,10,0.05,400
0,0.244026,0.245874,0.264383,-0.757008,-8.342171,2,2,100,0,10,0.03,300
0,0.274495,0.264129,0.264367,3.776376,3.689840,2,2,100,0,0,0.04,500
0,0.252433,0.249927,0.264287,0.992652,-4.695873,2,2,100,30,40,0.04,300


In [ ]:
# import lightgbm as lgb
# loaded_gbm = lgb.Booster(model_file='/home/mahaocheng/workspace/model_train/model_train_result/th_1_1010/train_result/score_th_3_1011.txt') 
# #loaded_gbm = lgb.Booster(model_file='/home/yutiangang/th_b7/model_train_result/score_th_b7_20250416_87feature_ol.txt') 
# DF_columns = loaded_gbm.feature_name()
# loaded_gbm.params

In [ ]:
# x_all = pd.concat([x_train,x_test,x_valid])
# y_all = pd.concat([y_train,y_test,y_valid])
# x_all.index=range(x_all.shape[0])
# y_all.index=range(y_all.shape[0])

In [64]:
##更新参数重新跑
model = LGBMClassifier(
    #Boosting Parameters参数
    objective='binary',          #binary:logistic – 二分类逻辑回归，输出为概率
    subsample=1,              #bagging_fraction，训练每个决策树所用到的子样本占总样本的比例；参考：0.8-1之间
    learning_rate=0.05,          #shrinkage rate 每次学习的步长. 越小模型越稳健  参考：[0.01, 0.02, 0.05, 0.1, 0.15,0.2]

    # other参数
    reg_alpha=0, 
    reg_lambda=10,                #L1 L2正则化项权重.可预防过拟合；
    scale_pos_weight=1,          #正样本的权重 默认为1； trick如果要加强负样本权重，此值设置<1??;
    seed=2024,                    #随机种子
    # is_unbalance=True,
    # sigmoid=1.0,                #sigmoid函数的参数,将用作于二分类;
    # subsample_for_bin=1000,     #构造分箱的样本数  默认50000;  手工1000 5000 10000 20000？？
    # subsample_freq=2,           #同一样本最大选择次数; 手工1 or 2？？
    # uniform_drop=False,         #only used in dart, true if want to use uniform drop
    # skip_drop=0.5,              #only used in dart, probability of skipping drop
    # xgboost_dart_mode=False,    #only used in dart, true if want to use xgboost dart mode
    # drop_rate=0.6,              #default=0.1, type=double,only used in dart
    verbose=-1,

    #Tree Parameters参数
    #max_bin=200,               #每个特征的最大分箱数. 默认255，手工加大？？
    max_depth=2,              #树的最大深度,一般5-8，推荐GBDT树的深度：6
    num_leaves=2,            #所有树最大叶子节点数. 默认=31
    #max_drop=5,               #最大叶子节点drop样本数  手工加大？？
    colsample_bytree=0.8,     #构造每棵树时的子样本抽样比例.  0.7-0.8
    min_child_samples = 100,    #构建一个叶子节点所需的最少样本量，alias=min_data_per_leaf , min_data, min_data_in_leaf. default=20  手工减小？？
    #min_child_weight=3,       #一个叶子节点所需最小权重和，默认0.001
    #min_split_gain=3.5,       #在叶子节点做进一步划分的最小损失，默认0
    n_estimators=400,         #决策树的数量  1500-2000
)
model.fit(x_train,y_train)
#model.fit(x_all,y_all)
result,predprob_train,predprob_test,predprob_valid=getKSresult(model,x_train,y_train,x_test,y_test,x_valid,y_valid)
result

,train_ks,test_ks,valid_ks,overfitting,loss
0,0.273544,0.258825,0.26566,5.380955,2.88196


### 保存入模变量

In [67]:
###入模变量及保存:
feature_importance_final = get_feature_importance(model, x_train, importance_type='gain')
feature_importance_final = feature_importance_final.reset_index()
feature_importance_final.columns = ['imp','feature']
print(feature_importance_final.shape)
columns=feature_importance_final.feature.to_list()

###入模变量保存
df_in_var = feature_importance_final
df_in_var.to_csv('/home/zengjunyao/notebook/to_zjy_模型训练/model_train_result/model_in_var_{}.csv'.format(datetime.now().strftime('%Y%m%d')),index=False)
print(df_in_var.shape)
#df_in_var.head(100)

(125, 2)
(125, 2)


In [68]:
df_in_var

,imp,feature
0,5711.687012,loan_id_base_v4_other_product_pre_payoff_amoun...
1,4705.933937,loan_id_base_v4_all_pre_payoff_amount_mean_rat...
2,3461.462936,fdc_id_base_v2_plt_agg_trans_amount_plt_sum_am...
3,2974.623047,fdc_id_base_v2_loan_days_over180d_heavy_overdu...
4,2745.837021,loan_id_base_v4_all_deny_pre_payoff_cnt_rate_i...
...,...,...
120,0.000000,app_id_comp_v1_seinst_updt_nonpreinst_iscomp_i...
121,0.000000,fdc_id_query_v1_history_query_cnt_diff_d30_d99999
122,0.000000,loan_id_base_v4_all_apply_amount_sum_max_in_ag...
123,0.000000,app_id_cate_v1_mininstalls_avg_y1


In [ ]:
#循环获取模型初筛后特征的Iv和Woe, 每个特征耗时0.5s
import math
save_interval = 100
col_list = feature_importance_final.feature.to_list()[:500]
loop_times = math.ceil(len(col_list)/save_interval)
print("特征数量: ",len(col_list))
print("保存间隔: ",save_interval)
print("循环次数: ",loop_times)

特征数量:  50
保存间隔:  100
循环次数:  1


In [ ]:
#保存：计算筛选后的变量train_woe, valid_woe
train_woe_result,valid_woe_result=saveTrainValidWoe(
    train_woes =train_woes,
    valid_woes = valid_woes,
    variables = iv_csi['feature'].to_list(),
    header=None,
    processMiss=True,
    addVar=True,
)
#合并 保存
woe_result = pd.concat([train_woe_result.reset_index(drop=True),emptyData(len(train_woe_result)),valid_woe_result.reset_index(drop=True)],axis=1)
woe_result.columns = ['variable_name', 'variable_cut', 'bad', 'good', 'badRate', 'Total',
       'good_pct', 'bad_pct', 'Total_pct', 'Odds', 'WOE', 'IV', 'space',
       'variable_name_oot', 'variable_cut_oot', 'bad_oot', 'good_oot', 'badRate_oot', 'Total_oot',
       'good_pct_oot', 'bad_pct_oot', 'Total_pct_oot', 'Odds_oot', 'WOE_oot', 'IV_oot']
woe_result.to_csv('model_train_result/all_feature_woe_{}_inmodel.csv'.format(datetime.now().strftime('%Y%m%d')),index = False,encoding='utf_8_sig')

# 样本打分

## 计算auc

In [69]:
#定义概率转分数的函数p2score
def p2score(prob):
    """
    概率转模型分
    """
    return round(550 - 60 / np.log(2) * np.log(prob / (1 - prob)), 0)

In [70]:
#类别预测：计算各数据集中每个样本的预测类别(0/1)，储存在predprob中
pred_train = pd.DataFrame(model.predict(x_train),columns=['predprob'])
pred_test = pd.DataFrame(model.predict(x_test),columns=['predprob'])
pred_valid = pd.DataFrame(model.predict(x_valid),columns=['predprob'])

In [71]:
#重置索引
x_train.index = range(x_train.shape[0])   #.shape[0]:返回行数 ; range(n):(0,n-1)
x_test.index = range(x_test.shape[0])
x_valid.index = range(x_valid.shape[0])
#x_valid = x_valid.reset_index(drop=True)  重置索引

train_data.index = range(train_data.shape[0])
test_data.index = range(test_data.shape[0])
valid_data.index = range(valid_data.shape[0])

In [72]:
#计算快，保存慢
#概率预测：预测样本为正类1的概率
pred_train = pd.DataFrame(model.predict_proba(x_train)[:,1],columns=['predprob'])  #model.predict_proba()：返回[属于类别0的概率,属于类别1的概率]
pred_test = pd.DataFrame(model.predict_proba(x_test)[:,1],columns=['predprob'])
pred_valid = pd.DataFrame(model.predict_proba(x_valid)[:,1],columns=['predprob'])
test_data['sample_type'] = 'test'

In [73]:
#将预测概率值与原始数据合并
try:
    train_data.drop(['predprob'], axis=1, inplace=True)
    test_data.drop(['predprob'], axis=1, inplace=True)
    valid_data.drop(['predprob'], axis=1, inplace=True)
except:
    pass
# 保留主键 target predprob score 将预测结果与原始数据合并
pred_train = train_data[["app_order_id",'sample_type',"label"] + df_in_var.feature.to_list()].join(pred_train)
pred_test = test_data[["app_order_id",'sample_type',"label"]+ df_in_var.feature.to_list()].join(pred_test)
pred_valid = valid_data[["app_order_id",'sample_type',"label"]+ df_in_var.feature.to_list()].join(pred_valid)
#按行拼接训练集、测试集、验证集的结果
pred_all = pd.concat([pred_train,pred_test,pred_valid],axis=0)  
# pred_all["apply_month"] = pred_all["date"].apply(lambda x: x[0:7]) 获取年月
# pred_all = pred_all.reset_index(drop=True)  重置索引
## 保存带概率的样本：
pred_all.to_pickle('/home/zengjunyao/notebook/to_zjy_模型训练/model_train_result/sample_data_with_pred_var_{}.pkl'.format(datetime.now().strftime('%Y%m%d')))
print(pred_all.shape)

(62913, 129)


,app_order_id,sample_type,label,loan_id_base_v4_other_product_pre_payoff_amount_sum_rate_in_90_d,loan_id_base_v4_all_pre_payoff_amount_mean_rate_in_3_orders,fdc_id_base_v2_plt_agg_trans_amount_plt_sum_amt_avg_d60,fdc_id_base_v2_loan_days_over180d_heavy_overdue_plt_cnt_ratio_d360,loan_id_base_v4_all_deny_pre_payoff_cnt_rate_in_30_d,loan_id_base_v4_all_pre_payoff_amount_sum_rate_in_15_d,loan_id_base_v4_all_payoff_inter_min_in_inf_d,loan_id_base_v4_other_product_pre_payoff_onloan_cnt_rate_in_90_d,fdc_id_base_v2_plt_agg_trans_amount_plt_sum_amt_std_d720,fdc_id_base_v2_cur_on_loan_cur_ovd_days_median_d360,loan_id_base_v4_other_product_pre_payoff_diff_range_in_7_orders,fdc_id_base_v2_large_trans_amount_over1m_old_mob_plt_plt_cnt_ratio_d30,loan_id_base_v4_brush_order_first_repay_early_day,loan_id_base_v4_all_payoff_pre_payoff_cnt_rate_in_180_d,fdc_id_base_v2_remain_amount_over70_light_overdue_plt_cnt_d720,fdc_id_base_v2_loan_quality_2_cur_ovd_days_avg_d360,fdc_id_base_v2_loan_days_over120d_old_mob_plt_plt_cnt_ratio_d30,fdc_id_base_v2_cur_on_loan_hist_ovd_days_median_d99999,ui_id_base_v2_user_education_level,app_id_gcate_v3_update_finance_mean_mininstalls_30d_60d_diff,fdc_id_base_v2_cash_loan_cur_ovd_days_sum_d180,loan_id_base_v4_all_pre_payoff_amount_mean_rate_in_15_d,fdc_id_base_v2_loan_days_over120d_early_settle_plt_cnt_ratio_d270,app_id_gcate_v3_update_other_var_ratings_infd,sms_id_base_v2_all_phone_max_body_discnt_groupby_phone_180d,app_id_base_v3_install_mean_app_cnt_groupby_date_3d,loan_id_base_v4_other_product_pre_payoff_amount_mean_rate_in_3_orders,fdc_id_base_v2_on_loan_remain_days_sum_d3,app_id_gcate_v3_update_other_var_score_90d,loan_id_base_v4_other_product_pre_payoff_cnt_in_7_orders,sms_id_loan_v1_repy_cnt,loan_id_base_v4_all_onloan_inter_max_in_7_orders,app_id_base_v3_install_mean_app_cnt_groupby_date_7d,app_id_base_v3_app_update_cnt,fdc_id_base_v2_loan_days_over120d_loan_quality_1_plt_cnt_ratio_d60,loan_id_base_v4_all_pre_payoff_overdue_days_mean_rate_in_7_orders,loan_id_base_v4_all_loan_success_amount_mean_in_3_d,fdc_id_base_v2_earliest_trans_plt_trans_amt_sum_d30,loan_id_base_v4_other_product_pre_payoff_cnt_in_15_orders,fdc_id_base_v2_cash_loan_remain_amt_median_d60,sms_id_base_v2_letter_phone_mean_word_discnt_groupby_phone_15d,loan_id_base_v4_all_pass_amount_mean_rate_in_15_orders,fdc_id_base_v2_loan_level_3_loan_cnt_ratio_d180,fdc_id_base_v2_large_trans_amount_over1m_early_settle_plt_cnt_ratio_d360,fdc_id_base_v2_loan_days_over180d_earliest_trans_plt_plt_cnt_ratio_d3,fdc_id_base_v2_light_overdue_remain_days_std_d99999,loan_id_base_v4_all_apply_amount_sum_range_in_agg_infd,...,app_id_comp_v1_seupdt_updt_nonpreinst_iscomp_updt_end_start_diff_m6_0,fdc_id_query_v1_history_query_cnt_rto_d3_d7,fdc_id_base_v2_loan_days_over180d_cur_on_loan_plt_plt_cnt_ratio_d99999,app_id_gcate_v3_update_life_var_ratings_15d_30d_diff,app_id_base_v3_install_app_cnt_3d,sms_id_base_v2_all_phone_var_body_discnt_groupby_phone_type_infd,loan_id_base_v4_self_product_deny_inter_mean_rate_in_inf_d,fdc_id_base_v2_most_trans_hist_ovd_days_max_d270,app_id_gcate_v3_update_finance_mean_mininstalls_30d_60d_ratio,loan_id_base_v4_other_app_loan_success_order_cnt_mean_in_agg_15d,app_id_cate_v1_cate_finance_reviews_avg_y1,sms_id_base_v2_digit_phone_word_discnt_ratio_180d,fdc_id_base_v2_loan_days_over60d_loan_level_3_plt_cnt_ratio_d720,sms_id_base_v2_digit_phone_max_body_discnt_groupby_date_180d,fdc_id_base_v2_loan_days_over180d_old_mob_plt_loan_cnt_ratio_d720,loan_id_base_v4_all_payoff_pre_payoff_cnt_rate_in_15_d,fdc_id_base_v2_heavy_overdue_remain_amt_sum_ratio_d99999,fdc_id_base_v2_hist_ovd_days_std_d360,fdc_id_base_v2_remain_amount_over90_light_overdue_loan_cnt_d99999,fdc_id_base_v2_new_mob_plt_remain_amt_min_d30,fdc_id_base_v2_light_overdue_trans_amt_max_ratio_d720,loan_id_base_v4_all_payoff_diff_mean_in_90_d,fdc_id_base_v2_remain_amount_over90_loan_level_3_loan_cnt_ratio_d270,fdc_id_base_v2_on_loan_remain_days_std_d720,fdc_id_base_v2_large_trans_amount_

In [74]:
#计算ks
print('训练集ks：',round(cal_ks(pred_train['predprob'].values,train_data['label'].values) * 100,2)) #cal_ks(预测概率值，真实标签类别值)
print('测试集ks：',round(cal_ks(pred_test['predprob'].values,test_data['label'].values) * 100,2))
print('验证集ks：',round(cal_ks(pred_valid['predprob'].values,pred_valid['label'].values) * 100,2))
#计算auc
print("    ")
print('训练集AUC:',round(roc_auc_score(y_train, pred_train[["predprob"]]),2))   #roc_auc_score(真实标签类别,预测概率值)
print('测试集AUC:',round(roc_auc_score(y_test, pred_test[["predprob"]]),2))
print('验证集AUC:',round(roc_auc_score(y_valid, pred_valid[["predprob"]]),2))

训练集ks： 27.35
测试集ks： 25.88
验证集ks： 26.57
    
训练集AUC: 0.69
测试集AUC: 0.68
验证集AUC: 0.67


## 统计psi

## 统计排序性

In [75]:
model.booster_.save_model('/home/zengjunyao/notebook/to_zjy_模型训练/model_train_result/score_af_wf_1027.txt')